In [2]:
import pandas as pd
import numpy as np
import pickle
import warnings
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')
 
from gensim import corpora
from gensim.models import LdaModel
 
print("Libraries imported!")

Libraries imported!


In [3]:
POS    = '#1D9E75'
NEU    = '#EF9F27'
NEG    = '#D85A30'
COLORS = ['#3B8BD4', '#1D9E75', '#EF9F27', '#D85A30', '#7F77DD']
 
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'axes.spines.top':   False,
    'axes.spines.right': False,
})


In [5]:
df = pd.read_csv('cleaned_reviews.csv')
print(f"Loaded {len(df)} reviews")

Loaded 1162 reviews


In [6]:
texts = [str(text).split() for text in df['clean_text'].dropna()]
print(f"Total documents: {len(texts)}")

Total documents: 1162


In [7]:
dictionary = corpora.Dictionary(texts)
print(f"Vocabulary before filter: {len(dictionary)}")

Vocabulary before filter: 6272


In [8]:
dictionary.filter_extremes(no_below=3, no_above=0.6)
print(f"Vocabulary after filter:  {len(dictionary)}")

Vocabulary after filter:  2693


In [9]:
corpus = [dictionary.doc2bow(text) for text in texts]
print(f"Corpus ready: {len(corpus)} documents")

Corpus ready: 1162 documents


In [11]:
print("Training LDA model... (takes ~15 seconds, please wait!)")
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,
    random_state=42,
    passes=10,
    alpha='auto'
)
print("LDA Training complete!")

Training LDA model... (takes ~15 seconds, please wait!)
LDA Training complete!


In [12]:
theme_names = {
    0: 'Headphones & Audio',
    1: 'Kindle & Reading',
    2: 'Fire TV & Streaming',
    3: 'Fire Tablet & Devices',
    4: 'Echo & Alexa',
}
 
print("\n" + "=" * 55)
print("THEMES DISCOVERED BY LDA:")
print("=" * 55)
for i, topic in lda_model.print_topics(num_words=8):
    words = [w.split('"')[1] for w in topic.split('+')]
    print(f"Theme {i+1} ({theme_names[i]}): {', '.join(words)}")


THEMES DISCOVERED BY LDA:
Theme 1 (Headphones & Audio): headphones, like, sound, dont, im, ive, people, nice
Theme 2 (Kindle & Reading): fire, kindle, one, amazon, new, paperwhite, like, screen
Theme 3 (Fire TV & Streaming): tv, prime, content, want, roku, amazon, great, use
Theme 4 (Fire Tablet & Devices): kindle, fire, tablet, hdx, hd, device, amazon, ipad
Theme 5 (Echo & Alexa): tap, echo, great, speaker, alexa, use, sound, amazon


In [13]:
def get_dominant_theme(text):
    bow    = dictionary.doc2bow(str(text).split())
    topics = lda_model.get_document_topics(bow)
    if not topics:
        return -1, 'Unknown'
    top_id = max(topics, key=lambda x: x[1])[0]
    return top_id, theme_names.get(top_id, f'Theme {top_id}')
 
df['theme_id'], df['theme_name'] = zip(*df['clean_text'].apply(get_dominant_theme))
 
print("Theme assigned to every review!")
print("Theme distribution:")
print(df['theme_name'].value_counts().to_string())

Theme assigned to every review!
Theme distribution:
theme_name
Echo & Alexa             590
Kindle & Reading         257
Headphones & Audio       135
Fire Tablet & Devices     98
Fire TV & Streaming       82


In [14]:
topic_data = []
for i, topic in lda_model.print_topics(num_words=8):
    pairs = [p.strip() for p in topic.split('+')]
    words_scores = []
    for p in pairs:
        score, word = p.split('*')
        words_scores.append((word.strip().strip('"'), float(score)))
    topic_data.append(words_scores)
 
json.dump({'topic_data': topic_data}, open('lda_results.json', 'w'))
 

In [15]:
with open('lda_model.pkl', 'wb') as f:
    pickle.dump(lda_model, f)
with open('lda_dict.pkl', 'wb') as f:
    pickle.dump(dictionary, f)
 
df.to_csv('final_reviews.csv', index=False)
 
print("Saved: lda_model.pkl, lda_dict.pkl, final_reviews.csv")

Saved: lda_model.pkl, lda_dict.pkl, final_reviews.csv


In [16]:
counts = df['theme_id'].value_counts().sort_index()
labels = [theme_names[i] for i in counts.index]
sizes  = counts.values
 
fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    sizes, colors=COLORS,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(width=0.6, edgecolor='white', linewidth=2),
    pctdistance=0.78
)
for at in autotexts:
    at.set(fontsize=10, fontweight='bold', color='white')
 
patches = [
    mpatches.Patch(color=COLORS[i], label=f'{labels[i]}  ({sizes[i]})')
    for i in range(len(labels))
]
ax.legend(handles=patches, loc='lower center',
          bbox_to_anchor=(0.5, -0.18), ncol=2, frameon=False, fontsize=9)
ax.set_title('What Topics Do Customers Talk About?',
             fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('chartE_themes.png', dpi=150, bbox_inches='tight')
plt.show()
print("chartE_themes.png SAVED!")

chartE_themes.png SAVED!


In [17]:
fig, axes = plt.subplots(1, 5, figsize=(14, 4))
 
for i, (ax, pairs) in enumerate(zip(axes, topic_data)):
    words  = [p[0] for p in pairs[:7]]
    scores = [p[1] for p in pairs[:7]]
    ax.barh(words[::-1], scores[::-1], color=COLORS[i], edgecolor='white')
    ax.set_title(theme_names[i], fontsize=9, fontweight='bold', color=COLORS[i])
    ax.set_xlabel('Weight', fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
 
plt.suptitle('Top Keywords in Each Discovered Theme',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('chartF_theme_keywords.png', dpi=150, bbox_inches='tight')
plt.show()
print("chartF_theme_keywords.png SAVED!")

chartF_theme_keywords.png SAVED!


In [18]:
fig, ax = plt.subplots(figsize=(9, 4.5))
 
theme_sent = df.groupby(['theme_id', 'sentiment']).size().unstack(fill_value=0)
theme_sent.index = [theme_names[i] for i in theme_sent.index]
 
bottom = np.zeros(len(theme_sent))
for sent, color, label in [
    ('positive', POS, 'Positive'),
    ('neutral',  NEU, 'Neutral'),
    ('negative', NEG, 'Negative'),
]:
    if sent in theme_sent.columns:
        vals = theme_sent[sent].values
        ax.bar(theme_sent.index, vals, bottom=bottom,
               color=color, label=label, edgecolor='white', linewidth=0.8)
        for j, (v, b) in enumerate(zip(vals, bottom)):
            if v > 10:
                ax.text(j, b + v/2, str(v),
                        ha='center', va='center',
                        fontsize=9, fontweight='bold', color='white')
        bottom += vals
 
ax.set_ylabel('Number of Reviews', fontsize=11)
ax.set_title('Sentiment Mix Inside Each Topic',
             fontsize=13, fontweight='bold')
ax.legend(frameon=False, fontsize=10)
plt.xticks(rotation=15, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('chartG_theme_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print("chartG_theme_sentiment.png SAVED!")

chartG_theme_sentiment.png SAVED!
